# Attention 教学版

这个 notebook 用来讲清标准 attention、FlashAttention 和融合思路之间的关系。

它的目标不是做完整实现，而是让读者先看懂数据流、成本和优化方向。

## 阅读顺序

1. 先看标准 attention 的 Q/K/V 数据流
2. 再看为什么 score / softmax 会放大中间张量
3. 然后看 FlashAttention 为什么能减少写回
4. 最后看融合为什么能进一步压缩中间状态

## 标准公式

$$\text{Attention}(Q, K, V) = \text{softmax}(QK^T / \sqrt{d})V$$

这个表达式本身不复杂，真正昂贵的是中间结果的存储和搬运。

## 一个最小的形状例子

- batch = 1
- seq_len = 4
- head_dim = 8

这个例子足够小，适合观察中间张量是怎么出现的。

## 一个可手算的 toy example

下面把维度缩小到 2，先看清楚 attention 的数据流，再回到更大的形状。

In [ ]:
import numpy as np

Q = np.array([[[1.0, 0.0], [0.0, 1.0]]], dtype='float32')
K = np.array([[[1.0, 0.0], [0.0, 1.0]]], dtype='float32')
V = np.array([[[1.0, 2.0], [3.0, 4.0]]], dtype='float32')

scores = Q @ K.transpose(0, 2, 1)
scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
probs = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
out = probs @ V

np.set_printoptions(precision=4, suppress=True)
print('Q =\n', Q[0])
print('K =\n', K[0])
print('V =\n', V[0])
print('scores =\n', scores[0])
print('probs =\n', probs[0])
print('out =\n', out[0])

In [ ]:
import numpy as np

batch, seq_len, head_dim = 1, 4, 8
Q = np.random.randn(batch, seq_len, head_dim).astype('float32')
K = np.random.randn(batch, seq_len, head_dim).astype('float32')
V = np.random.randn(batch, seq_len, head_dim).astype('float32')

scores = Q @ K.transpose(0, 2, 1) / np.sqrt(head_dim)
scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
probs = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
out = probs @ V

print('scores shape:', scores.shape)
print('probs shape:', probs.shape)
print('out shape:', out.shape)

## 观察点

- `scores` 是中间大张量
- `probs` 也是中间结果
- `out` 只是最后一步输出，不是最贵的部分
- 序列一长，这两层中间状态都会变贵

FlashAttention 的方向，就是尽量不要把这些中间状态完整落到慢层内存。

## 迁移到优化思路

- baseline：先把标准 attention 看懂
- FlashAttention：再把计算改成块化和 IO-aware
- fused attention：最后把周边算子也一起收进来

对应到本教程，就是先理解问题，再理解结构，最后理解优化。